# 真实结果与资源预算的只读复核

接入C09、C10与C11。本节读取公开的汇总JSON和CSV，重新计算差值并生成统计图。没有账户登录、原始行情读取、云提交或自动续跑。公开数据未到齐时显示“待发布”，不把缺失当零。图表只使用明确完成、可核验的指标。

旧截止版因连续121日规则阻塞；本节对应新的`cutoff_20260325_qlib_missing_v1`版本，二者不混合统计。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'课程大纲.md').is_file())
import matplotlib.pyplot as plt
folder=root/'reports/qukernet_cloud_v3/cutoff_20260325_qlib_missing_v1'
path=folder/'status.json'
state=json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
print('状态：',state.get('status') if state else '待发布：尚无本版本实测汇总')
if state:
    display(pd.DataFrame([state.get('execution',{})]))


## 先算上限，再读实际账本

两臂的搜索与微调配额一致。QuKerNet额外800条初值评价线路，使两臂全部计算成本不相等。失败与未知提交也占预算；请求shots、有效返回读数、已核验线路和批次是不同数量。

基础协议默认3小时。本次原窗口已保留，用户另行授权仅完成固定配额，2026年11月15日14:30:41（北京时间）续跑子窗口实际激活。实际资源须合并父子窗口；不能把完整链逻辑线路数作为子窗口新提交数。下方预算构造示例仍演示原默认期限，不用于恢复真实任务。


In [ ]:
budget=pd.DataFrame({'arm':['qukernet','random','analytic_preflight'],
                     'planned_circuits':[3044,2244,4]})
budget['planned_shots']=budget.planned_circuits*512
assert budget.planned_circuits.sum()==5292
assert budget.planned_shots.sum()==2709504
display(budget)
fig,ax=plt.subplots(figsize=(7,3));ax.bar(budget.arm,budget.planned_circuits)
ax.set_ylabel('Planned probability circuits');ax.set_title('Budget ceiling — not completed work');plt.show()
if state:
    e=state.get('execution',{})
    for key,limit in [('requested_circuits',5292),('requested_shots',2709504),('batches',271)]:
        value=e.get(key)
        if value is not None: assert 0<=value<=limit

resources_path=folder/'resources.csv'
resources=pd.read_csv(resources_path) if resources_path.exists() else pd.DataFrame()
if not resources.empty:
    display(resources)
    windows=resources[resources.stage.isin(['parent','continuation'])]
    if len(windows)==2 and state:
        assert windows.circuits.sum()==state['execution']['requested_circuits']
        assert windows.requested_shots.sum()==state['execution']['requested_shots']
        assert windows.batches.sum()==state['execution']['batches']


## 已完成指标与配对比较

主指标按10个评价日期等权，不把50条证券记录当50个独立市场日期。只有完整50条、相同日期/样本/目标都匹配的端点进入预定配对比较。CSV只包含方法层汇总，没有证券代码、逐样本预测或角度。


In [ ]:
metrics_path=folder/'metrics.csv'
metrics=pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
if metrics.empty:
    print('尚无可发布的模型指标；不绘制效果曲线。')
else:
    display(metrics)
    ready=metrics[(metrics['split']=='evaluation') & (metrics['status']=='complete') & metrics['accuracy_date_equal'].notna()]
    if not ready.empty:
        order=[('qukernet','initial'),('qukernet','endpoint'),('random','endpoint'),('rbf','classical'),
               ('linear','classical'),('full_train_linear','classical'),('majority','classical'),('always0','classical'),('always1','classical')]
        labels=['QuKerNet\ninitial','QuKerNet\nfinetuned','Random\nfinetuned','RBF SVC',
                'Linear SVC\n16 samples','Linear SVC\nfull training','Majority','Always\ndown','Always\nup']
        lookup={(row.method,row.checkpoint):row.accuracy_date_equal for row in ready.itertuples()}
        picks=[(labels[i],lookup[key]) for i,key in enumerate(order) if key in lookup]
        fig,ax=plt.subplots(figsize=(9,4));bars=ax.bar([p[0] for p in picks],[p[1] for p in picks])
        ax.bar_label(bars,labels=[f'{p[1]:.0%}' for p in picks],padding=3)
        ax.set_ylim(0,1);ax.set_ylabel('Date-equal accuracy');ax.tick_params(axis='x',labelsize=8)
        ax.set_title('50 evaluation rows across 10 dates');plt.tight_layout();plt.show()
comparisons_path=folder/'comparisons.csv'
comparisons=pd.read_csv(comparisons_path) if comparisons_path.exists() else pd.DataFrame()
if not comparisons.empty:
    display(comparisons)
    for r in comparisons.itertuples():
        available=str(r.available).lower()=='true'
        if available:
            assert r.left_rows==r.right_rows==r.expected_rows==50
            assert np.isclose(r.difference,r.left_value-r.right_value)

# 只读取已发布的不确定性与日期级汇总；没有区间时不自行补造。
uncertainty_path=folder/'paired_uncertainty.csv'
uncertainty=pd.read_csv(uncertainty_path) if uncertainty_path.exists() else pd.DataFrame()
if not uncertainty.empty:
    display(uncertainty)
    primary=uncertainty[uncertainty.block_length==2]
    fig,ax=plt.subplots(figsize=(8,3))
    for i,row in enumerate(primary.itertuples()):
        ax.plot(row.estimate,i,'o');ax.hlines(i,row.ci_low,row.ci_high)
    ax.set_yticks(range(len(primary)),primary.comparison,fontsize=8)
    ax.axvline(0,color='gray');ax.set_xlabel('Paired date-equal accuracy difference');plt.tight_layout();plt.show()
date_path=folder/'date_metrics.csv'
date_metrics=pd.read_csv(date_path) if date_path.exists() else pd.DataFrame()
if not date_metrics.empty:
    assert date_metrics.n_rows.eq(5).all()
    print('日期索引保留配对顺序；不发布证券逐样本或实际私有日期。')
    display(date_metrics.groupby(['method','checkpoint']).accuracy.agg(['count','mean']))


## 缺失覆盖与结果边界

检查填补比例和有效支持数，区分流程通过与信息足够。KTA提高而预测未改善，只能说明训练代理目标提高。只有单种子、一个短评价窗、每股一次评价，不能证明普遍优越、未来盈利或量子优势。初值与终点各自使用独立512-shot核估计并重新拟合，单次差值混合参数更新和测量随机性。日期块bootstrap只重采样已得到的预测，不覆盖重新抽shots或搜索种子的不确定性；正差值不能直接解释为纯微调因果收益。即使执行停止时只完成部分阶段，完整账本和失败解释仍是有效研究产物。

日期块指10个已观察评价日期按时间排序后的相邻非循环块，不是连续市场交易日。每个日期5只不同股票，每股一次，因此区间只适用于该固定样本的探索性描述。
本轮每条线路512 shots的一次流程不验证重复测量选择稳定性或失选率；固定候选重复采样属于待检验研究，旧20股诊断不得合并为本轮50股证据。


In [ ]:
quality_path=folder/'data_quality.csv'
quality=pd.read_csv(quality_path) if quality_path.exists() else pd.DataFrame()
if not quality.empty:
    display(quality)
    if {'split','rows','rows_with_any_missing_80'}.issubset(quality.columns):
        fig,ax=plt.subplots(figsize=(6,3))
        ax.bar(quality.split,quality.rows_with_any_missing_80/quality.rows)
        ax.set_ylim(0,max(.01,float((quality.rows_with_any_missing_80/quality.rows).max())*1.4));ax.set_ylabel('Rows with missing raw features / rows');plt.show()
if state:
    print('结论：', state.get('conclusion','尚未形成效果判断'))
    print('本Notebook只重新读取汇总；不恢复运行。')

# 只读谱修正汇总；真实交叉核及模型参数留在私有记录中。
diagnostic_path=folder/'kernel_diagnostics.csv'
diagnostics=pd.read_csv(diagnostic_path) if diagnostic_path.exists() else pd.DataFrame()
if not diagnostics.empty:
    display(diagnostics)
    complete=diagnostics[diagnostics.status.eq('complete')]
    assert complete.frobenius.ge(0).all()
    cross=complete[complete.source.eq('posthoc_saved_cloud_cross_kernel')]
    if not cross.empty:
        fig,ax=plt.subplots(figsize=(8,3))
        ax.bar(cross.method+'/'+cross.checkpoint+'/'+cross['split'],cross.frobenius)
        ax.set_ylabel('Actual cross projection Frobenius norm')
        ax.tick_params(axis='x',rotation=25);plt.tight_layout();plt.show()


## 解析控制流：部分返回与重复预约

下面使用内存字典和JSON往返模拟持久化状态，不连接账户、不创建真实预约。第一次预约即计入全部请求；只返回部分结果时，不能把未返回线路当作完成，也不能重复提交原操作。重载沿用原截止时间。这个练习说明停止规则，不替代生产执行器的离线测试与真实账本。

In [ ]:
# 解析控制流演示：所有数字均为构造；不代表真实实验消耗。
import copy
ledger={'deadline':10800,'reserved':{},'stopped':False}
def reserve(book,operation,circuits,now):
    if book['stopped'] or now>=book['deadline']:
        raise RuntimeError('execution_stopped')
    if operation in book['reserved']:
        raise RuntimeError('duplicate_operation')
    book['reserved'][operation]={'circuits':circuits,'returned':0,'state':'reserved'}
def accept_partial(book,operation,returned,now):
    row=book['reserved'][operation]
    assert 0<=returned<=row['circuits']
    row['returned']=returned
    row['state']='complete' if returned==row['circuits'] else 'partial'
    if now>=book['deadline'] or row['state']=='partial':
        book['stopped']=True
reserve(ledger,'fixed-operation-001',20,0)
try:
    reserve(ledger,'fixed-operation-001',20,1)
except RuntimeError as exc:
    assert str(exc)=='duplicate_operation'
else:
    raise AssertionError('duplicate was not blocked')
accept_partial(ledger,'fixed-operation-001',12,300)
reloaded=json.loads(json.dumps(ledger))
assert reloaded['deadline']==10800 and reloaded['stopped']
assert sum(v['circuits'] for v in reloaded['reserved'].values())==20
assert sum(v['returned'] for v in reloaded['reserved'].values())==12
try:
    reserve(reloaded,'fixed-operation-002',20,301)
except RuntimeError as exc:
    assert str(exc)=='execution_stopped'
else:
    raise AssertionError('partial-return stop was bypassed')
print('构造检查通过：预约20、返回12；重复预约与停止后的新预约均被阻止。')
print('这些数字不写入真实资源账本；客户端停止不代表云后台取消。')


## 真实执行恢复的边界

含本次续跑子窗口的账本须由`cloud_v3.completion_ledger`读取，旧仅识别3小时窗口的执行器不能直接重启或新建实验。续跑默认单次且不自动恢复；新的实验需另建明确协议并完成账本准入。本Notebook没有真实执行开关，只读取已保存汇总。

训练谱修正取保存记录；验证/评价交叉核修正用已核验读数重建B后计算‖B−(BU)Uᵀ‖F，属于新增事后诊断。拟合时空交叉块给出的0不能当作真实V/O修正幅度；没有完整证据的行保持unknown。
